## Test if CUDA is installed with PyTorch


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce 940MX


## Create directories and set filenames

In [2]:
import os 



video_path = "data/can/video.mp4"
image_dir = "data/can/frames"
workspace = "data/can/workspace"
database = workspace + "database.db"
sparse = workspace + "/sparse"
dense = workspace + "/dense"
ply_output = workspace + "/pointcloud.ply"
#maximagesize is low here because my GPU is old
max_image_size = 500

os.makedirs(sparse, exist_ok=True)
os.makedirs(dense, exist_ok=True)

## Extract Frames from Video

In [3]:
import cv2
import os

#not every frame of the video has to be used 
step = 5

os.makedirs(image_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

i = 0
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if i % step == 0:
        cv2.imwrite(f"{image_dir}/{frame_id:05d}.jpg", frame)
        frame_id += 1

    i += 1

cap.release()
print(f"Extracted {frame_id} frames")

Extracted 46 frames


## Extract Features 

In [4]:
!colmap feature_extractor \
 --database_path {database} \
 --image_path {image_dir} \
 --ImageReader.single_camera 1 \
 --SiftExtraction.max_image_size {max_image_size}


Feature extraction

Processed file [1/46]
  Name:            00000.jpg
  Dimensions:      576 x 1024
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1228.80px
  Features:        721
Processed file [2/46]
  Name:            00001.jpg
  Dimensions:      576 x 1024
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1228.80px
  Features:        862
Processed file [3/46]
  Name:            00002.jpg
  Dimensions:      576 x 1024
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1228.80px
  Features:        701
Processed file [4/46]
  Name:            00003.jpg
  Dimensions:      576 x 1024
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1228.80px
  Features:        723
Processed file [5/46]
  Name:            00004.jpg
  Dimensions:      576 x 1024
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1228.80px
  Features:        833
Processed file [6/46]
  Name:            00005.jpg
  Dimensions:      576 x 1024
  Camera:          #1 - SIMPLE_RADIAL
  Foc

## Match Features

In [5]:
!colmap exhaustive_matcher \
  --database_path {database}


Exhaustive feature matching

Matching block [1/1, 1/1] in 2.926s
Elapsed time: 0.051 [minutes]


## Mapper Step (build sparse reconstruction)

In [6]:
!colmap mapper   \
--database_path {database}  \
 --image_path {image_dir} \
   --output_path {sparse}


Loading database

Loading cameras... 1 in 0.000s
Loading matches... 956 in 0.006s
Loading images... 46 in 0.004s (connected 46)
Building correspondence graph... in 0.025s (ignored 0)

Elapsed time: 0.001 [minutes]


Finding good initial image pair


Initializing with image pair #6 and #11


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  2.900067e+04    0.00e+00    2.33e+05   0.00e+00   0.00e+00  1.00e+04        0    6.54e-03    1.54e-02
   1  7.892212e+07   -7.89e+07    2.33e+05   8.81e+02  -3.47e+03  5.00e+03        1    3.34e-03    1.88e-02
   2  2.966851e+07   -2.96e+07    2.33e+05   5.46e+02  -1.34e+03  1.25e+03        1    1.96e-04    1.90e-02
   3  1.280314e+07   -1.28e+07    2.33e+05   1.01e+02  -6.18e+02  1.56e+02        1    1.48e-04    1.92e-02
   4  4.316539e+05   -4.03e+05    2.33e+05   4.05e+01  -2.47e+01  9.77e+00        1    1.38e-04    1.93e-02
   5  2.215871e+04    6.84e+03   

## Undistort images 

In [7]:
!colmap image_undistorter \
--image_path {image_dir}  \
--input_path {sparse}/0  \
--output_path {dense}  \
--output_type COLMAP \
--max_image_size {max_image_size}


Reading reconstruction

 => Reconstruction with 46 images and 2203 points

Image undistortion

Undistorting image [1/46]
Undistorting image [2/46]
Undistorting image [3/46]
Undistorting image [4/46]
Undistorting image [5/46]
Undistorting image [6/46]
Undistorting image [7/46]
Undistorting image [8/46]
Undistorting image [9/46]
Undistorting image [10/46]
Undistorting image [11/46]
Undistorting image [12/46]
Undistorting image [13/46]
Undistorting image [14/46]
Undistorting image [15/46]
Undistorting image [16/46]
Undistorting image [17/46]
Undistorting image [18/46]
Undistorting image [19/46]
Undistorting image [20/46]
Undistorting image [21/46]
Undistorting image [22/46]
Undistorting image [23/46]
Undistorting image [24/46]
Undistorting image [25/46]
Undistorting image [26/46]
Undistorting image [27/46]
Undistorting image [28/46]
Undistorting image [29/46]
Undistorting image [30/46]
Undistorting image [31/46]
Undistorting image [32/46]
Undistorting image [33/46]
Undistorting image [34

In [8]:
!colmap patch_match_stereo  \
--workspace_path {dense} \
--workspace_format COLMAP \
--PatchMatchStereo.geom_consistency true \
--PatchMatchStereo.max_image_size {max_image_size}


Reading workspace...
Reading configuration...
Configuration has 46 problems...

Processing view 1 / 46 for 00000.jpg

Reading inputs...

PatchMatch::Problem
-------------------
ref_image_idx: 0
src_image_idxs: 10 2 3 5 4 6 7 8 9 11 22 21 23 33 12 28 27 24 32 20

PatchMatchOptions
-----------------
max_image_size: 500
gpu_index: 0
depth_min: 4.36847
depth_max: 13.8421
window_radius: 5
window_step: 1
sigma_spatial: 5
sigma_color: 0.2
num_samples: 15
ncc_sigma: 0.6
min_triangulation_angle: 1
incident_angle_sigma: 0.9
num_iterations: 5
geom_consistency: 0
geom_consistency_regularizer: 0.3
geom_consistency_max_cost: 3
filter: 0
filter_min_ncc: 0.1
filter_min_triangulation_angle: 3
filter_min_num_consistent: 2
filter_geom_consistency_max_cost: 1
write_consistency_graph: 0
allow_missing_files: 0

PatchMatch::Run
---------------
Initialization: 0.0888s
 Sweep 1: 0.4073s
 Sweep 2: 0.3060s
 Sweep 3: 0.4190s
 Sweep 4: 0.3137s
Iteration 1: 1.4470s
 Sweep 1: 0.3993s
 Sweep 2: 0.3083s
 Sweep 3: 0.40

In [9]:
!colmap stereo_fusion \
 --workspace_path {dense} \
 --workspace_format COLMAP \
 --input_type geometric \
 --output_path {ply_output} 


StereoFusion::Options
---------------------
mask_path: 
max_image_size: -1
min_num_pixels: 5
max_num_pixels: 10000
max_traversal_depth: 100
max_reproj_error: 2
max_depth_error: 0.01
max_normal_error: 10
check_num_images: 50
use_cache: 0
cache_size: 32
bbox_min: -3.40282e+38 -3.40282e+38 -3.40282e+38
bbox_max: 3.40282e+38 3.40282e+38 3.40282e+38

Reading workspace...
Loading workspace data with 4 threads...
Elapsed time: 0.009 [minutes]
Reading configuration...
Starting fusion with 4 threads
Fusing image [1/46] with index 0 in 3.290s (27092 points)
Fusing image [2/46] with index 1 in 0.367s (32533 points)
Fusing image [3/46] with index 2 in 0.383s (37285 points)
Fusing image [4/46] with index 3 in 0.261s (40742 points)
Fusing image [5/46] with index 5 in 0.326s (45916 points)
Fusing image [6/46] with index 4 in 0.126s (47474 points)
Fusing image [7/46] with index 6 in 0.217s (50927 points)
Fusing image [8/46] with index 7 in 0.221s (54088 points)
Fusing image [9/46] with index 8 in 0.2

## Visualize PointCloud in Open3d

In [10]:
import open3d as o3d

# Load PLY file
pcd = o3d.io.read_point_cloud(ply_output)

# Optional: print info
print(pcd)

# Visualize
o3d.visualization.draw_geometries([pcd])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PointCloud with 124295 points.
